# Chapter 9 — Agentic Formative Assessment: Results and Figures

**Companion notebook | Paired evaluation, expert ratings, and publication figures**

This notebook reconstructs the numerical summaries and seven experimental figures discussed in Chapter 9, *Agentic AI for Automated Formative Assessment and Adaptive Learning Support*. It uses the **anonymised, case-level tabulations embedded in the supplied results notebook**. No original mathematical questions, learner responses, generated feedback texts, or reviewer identities are included.

The figures examine three distinct questions: whether each configuration returns a complete assessment draft, how available drafts are rated for mathematical and pedagogical quality, and how computational operations and processing time differ. The analysis **does not measure student learning gains** and **does not run the assessment-generation pipelines**.

**Evaluation design**

- **Held-out operational assessment:** 76 matched mathematics problems, processed by F1 and F2.1 (152 system–problem records).
- **Independent expert assessment:** 30 selected problems; 58 available drafts; 28 matched problem pairs rated by one expert on four 0–2 criteria.
- **Exploratory development assessment:** 11 shared problems compared across F1, F2.1, and F2.2 using three development review forms. Development results are reported separately.
- Inference over mathematical operations establishes **numerical outcomes**, not the semantic suitability of the operation or the educational quality of feedback. All generated drafts required teacher review.

**Data availability and scope:** The provided notebook contains anonymised numerical records and reproduces the reported numerical results. The original held-out inputs, model-generated responses, and source review sheets are not included. Thus, the **analysis and visualisations can be rerun**, but the original experiment and the derivation of these tabulations **cannot be independently audited from this notebook alone**. The full held-out set should not be described as entirely sourced from MathDial without separate documentary evidence.

**Requirements:** Python 3.10 or later, `numpy`, `matplotlib`, `scipy`, and `IPython`. Install missing packages in the active notebook environment using `%pip install numpy matplotlib scipy ipython`.

**Output:** Seven figures saved in `chapter9_figures/` as PNG (300 dpi), SVG, and vector PDF. The original source figure names are preserved.

| Figure in the book | Export filename (without extension) | Analysis |
|---|---|---|
| 9.2 | `Fig07_exploratory_development_ratings` | Exploratory development comparison |
| 9.3 | `Fig01_heldout_response_coverage` | Held-out output coverage |
| 9.4 | `Fig02_expert_rubric_comparison` | Four expert-rating dimensions |
| 9.5 | `Fig03_paired_critical_error_comparison` | Critical-error flags |
| 9.6 | `Fig04_expert_paired_score_differences` | Paired quality-score differences |
| 9.7 | `Fig05_execution_time_tradeoff` | Execution-time comparison |
| 9.8 | `Fig06_arithmetic_operation_states` | Arithmetic operation states |


<div style="border:1px solid #d8e0ea;border-left:4px solid #4c6b8a;border-radius:7px;background:#f7f9fc;padding:10px 14px;margin:13px 0 8px 0;">
<strong>01 | Imports and figure settings</strong><div style="color:#46596d;margin-top:5px;font-size:0.94em;">Configure reproducible numerical calculations, a consistent visual style, and figure exports.</div>
</div>

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
import json
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon, binomtest
from IPython.display import display
import hashlib

OUTPUT_DIR = Path("chapter9_figures")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

COLORS = {"F1": "#53677B", "F2.1": "#087F78", "F2.2": "#B68A45"}
RED = "#C15A50"
PALE = "#D8E2E8"
plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 9.5,
    "axes.titlesize": 11, "axes.titleweight": "semibold",
    "axes.labelsize": 9.5, "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": "#80909D", "axes.labelcolor": "#23313B",
    "text.color": "#23313B", "xtick.color": "#485965",
    "ytick.color": "#485965", "grid.color": "#E4E9ED",
    "savefig.facecolor": "white", "figure.facecolor": "white",
    "pdf.fonttype": 42, "svg.fonttype": "none",
})

def export_figure(fig, name):
    for extension in ("png", "svg", "pdf"):
        fig.savefig(OUTPUT_DIR / f"{name}.{extension}",
                    dpi=300, bbox_inches="tight", pad_inches=0.15)
    display(fig)
    plt.close(fig)
    print("Saved:", name, "(PNG / SVG / PDF)")

def style_grid(ax, axis="y"):
    ax.grid(axis=axis, linewidth=0.75)
    ax.set_axisbelow(True)

RANDOM_SEED = 20261106


<div style="border:1px solid #d8e0ea;border-left:4px solid #4c6b8a;border-radius:7px;background:#f7f9fc;padding:10px 14px;margin:13px 0 8px 0;">
<strong>02 | Load the embedded analysis records</strong><div style="color:#46596d;margin-top:5px;font-size:0.94em;">Read the anonymised numerical tables. Original questions, reviewer identities, and private allocation files are not distributed.</div>
</div>

In [ ]:
# Anonymised numeric records from the supplied source notebook.
# Records are included for traceable analysis, not as a substitute for the source archive.
DATA = json.loads(r'''{"heldout":[{"arithmetic_status":"rejected","automatic_release":false,"case_id":"eval_000","diagnosis_available":true,"feedback_available":true,"method":"F1","model_calls":2,"review_required":true,"wall_seconds":117.413},{"arithmetic_status":"rejected","automatic_release":false,"case_id":"eval_001","diagnosis_available":true,"feedback_available":true,"method":"F1","model_calls":2,"review_required":true,"wall_seconds":101.862},{"arithmetic_status":"rejected","automatic_release":false,"case_id":"eval_002","diagnosis_available":true,"feedback_available":true,"method":"F1","model_calls":2,"review_required":true,"wall_seconds":125.294},{"arithmetic_status":"rejected","automatic_release":false,"case_id":"eval_003","diagnosis_available":true,"feedback_available":true,"method":"F1","model_calls":2,"review_required":true,"wall_seconds":88.997},{"arithmetic_status":"rejected","automatic_release":false,"case_id":"eval_004","diagnosis_available":true,"feedback_available":true,"method":"F1","model_calls":2,"review_required":true,"wall_seconds":87.318},{"arithmetic_status":"arithmetic_evaluated","automatic_release":false,"case_id":"eval_005","diagnosis_available":true,"feedback_available":true,"method":"F1","model_calls":2,"review_required":true,"wall_seconds":126.506},{"arithmetic_status":"rejected","automatic_release":false,"case_id":"eval_006","diagnosis_available":true,"feedback_available":true,"method":"F1","model_calls":2,"review_required":true,"wall_seconds":77.336},{"arithmetic_status":"arithmetic_evaluated","automatic_release":false,"case_id":"eval_007","diagnosis_available":true,"feedback_available":true,"method":"F1","model_calls":2,"review_required":true,"wall_seconds":104.062},{"arithmetic_status":"rejected","automatic_release":false,"case_id":"eval_008","diagnosis_available":true,"feedback_available":true,"method":"F1","model_calls":2,"review_required":true,"wall_seconds":133.448},{"arithmetic_status":"rejected","automatic_release":false,"case_id":"eval_009","diagnosis_available":true,"feedback_available":true,"method":"F1","model_calls":2,"review_required":true,"wall_seconds":105.464},{"arithmetic_status":"arithmetic_evaluated","automatic_release":false,"case_id":"eval_010","diagnosis_available":true,"feedback_available":true,"method":"F1","model_calls":2,"review_required":true,"wall_seconds":89.458},{"arithmetic_status":"rejected","automatic_release":false,"case_id":"eval_011","diagnosis_available":true,"feedback_available":true,"method":"F1","model_calls":2,"review_required":true,"wall_seconds":68.675},{"arithmetic_status":"not_attempted","automatic_release":false,"case_id":"eval_012","diagnosis_available":false,"feedback_available":false,"method":"F1","model_calls":1,"review_required":true,"wall_seconds":26.537},{"arithmetic_status":"arithmetic_evaluated","automatic_release":false,"case_id":"eval_013","diagnosis_available":true,"feedback_available":true,"method":"F1","model_calls":2,"review_required":true,"wall_seconds":83.892},{"arithmetic_status":"rejected","automatic_release":false,"case_id":"eval_014","diagnosis_available":true,"feedback_available":true,"method":"F1","model_calls":2,"review_required":true,"wall_seconds":83.579},{"arithmetic_status":"rejected","automatic_release":false,"case_id":"eval_015","diagnosis_available":true,"feedback_available":true,"method":"F1","model_calls":2,"review_required":true,"wall_seconds":77.182},{"arithmetic_status":"arithmetic_evaluated","automatic_release":false,"case_id":"eval_016","diagnosis_available":true,"feedback_available":true,"method":"F1","model_calls":2,"review_required":true,"wall_seconds":107.067},{"arithmetic_status":"not_attempted","automatic_release":false,"case_id":"eval_017","diagnosis_available":false,"feedback_available":false,"method":"F1","model_calls":1,"review_required":true,"wall_seconds":27.127},{"arithmetic_status":"rejected","automatic_release":false,"case_id":"eval_018","diagnosis_available":true,"feedback_available":true,"method":"F1","model_calls":2,"review_required":true,"wall_seconds":99.578},{"arithmetic_status":"rejected","automatic_release":false,"case_id":"eval_019","diagnosis_available":true,"feedback_available":true,"method":"F1","model_calls":2,"review_required":true,"wall_seconds":90.894},{"arithmetic_status":"rejected","automatic_release":false,"case_id":"eval_020","diagnosis_available":true,"feedback_available":true,"method":"F1","model_calls":2,"review_required":true,"wall_seconds":83.122},{"arithmetic_status":"rejected","automatic_release":false,"case_id":"eval_021","diagnosis_available":true,"feedback_available":true,"method":"F1","model_calls":2,"review_required":true,"wall_seconds":123.777},{"arithmetic_status":"rejected","automatic_release":false,"case_id":"eval_022","diagnosis_available":true,"feedback_available":true,"method":"F1","model_calls":2,"review_required":true,"wall_seconds":80.59},{"arithmetic_status":"rejected","automatic_release":false,"case_id":"eval_023","diagnosis_available":true,"feedback_available":true,"method":"F1","model_calls":2,"review_required":true,"wall_seconds":115.689},{"arithmetic_status":"rejected","automatic_release":false,"case_id":"eval_024","diagnosis_available":true,"feedback_available":true,"method":"F1","model_calls":2,"review_required":true,"wall_seconds":124.586},{"arithmetic_status":"rejected","automatic_release":false,"case_id":"eval_025","diagnosis_available":true,"feedback_available":true,"method":"F1","model_calls":2,"review_required":true,"wall_seconds":69.312},{"arithmetic_status":"rejected","automatic_release":false,"case_id":"eval_026","diagnosis_available":true,"feedback_available":true,"method":"F1","model_calls":2,"review_required":true,"wall_seconds":80.014},{"arithmetic_status":"not_attempted","automatic_release":false,"case_id":"eval_027","diagnosis_available":false,"feedback_available":false,"method":"F1","model_calls":1,"review_required":true,"wall_seconds":39.105},{"arithmetic_status":"rejected","automatic_release":false,"case_id":"eval_028","diagnosis_available":true,"feedback_available":true,"method":"F1","model_calls":2,"review_required":true,"wall_seconds":67.095},{"arithmetic_status":"not_proposed","automatic_release":false,"case_id":"eval_029","diagnosis_available":true,"feedback_available":true,"method":"F1","model_calls":2,"review_required":true,"wall_seconds":83.321},{"arithmetic_status":"rejected","automatic_release":false,"case_id":"eval_030","diagnosis_available":true,"feedback_available":true,"method":"F1","model_calls":2,"review_required":true,"wall_seconds":106.224},{"arithmetic_status":"arithmetic_evaluated","automatic_release":false,"case_id":"eval_031","diagnosis_available":true,"feedback_available":true,"method":"F1","model_calls":2,"review_required":true,"wall_seconds":111.808},{"arithmetic_status":"not_attempted","automatic_release":false,"case_id":"eval_032","diagnosis_available":false,"feedback_available":false,"method":"F1","model_calls":0,"review_required":true,"wall_seconds":147.8},{"arithmetic_status":"rejected","automatic_release":false,"case_id":"eval_033","diagnosis_available":true,"feedback_available":true,"method":"F1","model_calls":2,"review_required":true,"wall_seconds":76.016},{"arithmetic_status":"arithmetic_evaluated","automatic_release":false,"case_id":"eval_034","diagnosis_available":true,"feedback_available":true,"method":"F1","model_calls":2,"review_required":true,"wall_seconds":90.428},{"arithmetic_status":"rejected","automatic_release":false,"case_id":"eval_035","diagnosis_available":true,"feedback_available":true,"method":"F1","model_calls":2,"review_required":true,"wall_seconds":83.592},{"arithmetic_status":"rejected","automatic_release":false,"case_id":"eval_036","diagnosis_available":true,"feedback_available":true,"method":"F1","model_calls":2,"review_required":true,"wall_seconds":106.501},{"arithmetic_status":"not_attempted","automatic_release":false,"case_id":"eval_037","diagnosis_available":false,"feedback_available":false,"method":"F1","model_calls":1,"review_required":true,"wall_seconds":39.622},{"arithmetic_status":"arithmetic_evaluated","automatic_release":false,"case_id":"eval_038","diagnosis_available":true,"feedback_available":true,"method":"F1","model_calls":2,"review_required":true,"wall_seconds":118.399},{"arithmetic_status":"arithmetic_evaluated","automatic_release":false,"case_id":"eval_039","diagnosis_available":true,"feedback_available":true,"method":"F1","model_calls":2,"review_required":true,"wall_seconds":130.19},{"arithmetic_status":"rejected","automatic_release":false,"case_id":"eval_040","diagnosis_available":true,"feedback_available":true,"method":"F1","model_calls":2,"review_required":true,"wall_seconds":81.305},{"arithmetic_status":"rejected","automatic_release":false,"case_id":"eval_041","diagnosis_available":true,"feedback_available":true,"method":"F1","model_calls":2,"review_required":true,"wall_seconds":122.12},{"arithmetic_status":"rejected","automatic_release":false,"case_id":"eval_042","diagnosis_available":true,"feedback_available":true,"method":"F1","model_calls":2,"review_required":true,"wall_seconds":127.154},{"arithmetic_status":"rejected","automatic_release":false,"case_id":"eval_043","diagnosis_available":true,"feedback_available":true,"method":"F1","model_calls":2,"review_required":true,"wall_seconds":72.539},{"arithmetic_status":"rejected","automatic_release":false,"case_id":"eval_044","diagnosis_available":true,"feedback_available":true,"method":"F1","model_calls":2,"review_required":true,"wall_seconds":71.387},{"arithmetic_status":"rejected","automatic_release":false,"case_id":"eval_045","diagnosis_available":true,"feedback_available":true,"method":"F1","model_calls":2,"review_required":true,"wall_seconds":81.82},{"arithmetic_status":"not_proposed","automatic_release":false,"case_id":"eval_046","diagnosis_available":true,"feedback_available":true,"method":"F1","model_calls":2,"review_required":true,"wall_seconds":97.437},{"arithmetic_status":"rejected","automatic_release":false,"case_id":"eval_047","diagnosis_available":true,"feedback_available":true,"method":"F1","model_calls":2,"review_required":true,"wall_seconds":84.325},{"arithmetic_status":"rejected","automatic_release":false,"case_id":"eval_048","diagnosis_available":true,"feedback_available":true,"method":"F1","model_calls":2,"review_required":true,"wall_seconds":97.868},{"arithmetic_status":"arithmetic_evaluated","automatic_release":false,"case_id":"eval_049","diagnosis_available":true,"feedback_available":true,"method":"F1","model_calls":2,"review_required":true,"wall_seconds":109.954},{"arithmetic_status":"rejected","automatic_release":false,"case_id":"eval_050","diagnosis_available":true,"feedback_available":true,"method":"F1","model_calls":2,"review_required":true,"wall_seconds":95.575},{"arithmetic_status":"arithmetic_evaluated","automatic_release":false,"case_id":"eval_051","diagnosis_available":true,"feedback_available":true,"method":"F1","model_calls":2,"review_required":true,"wall_seconds":63.62},{"arithmetic_status":"rejected","automatic_release":false,"case_id":"eval_052","diagnosis_available":true,"feedback_available":true,"method":"F1","model_calls":2,"review_required":true,"wall_seconds":79.717},{"arithmetic_status":"arithmetic_evaluated","automatic_release":false,"case_id":"eval_053","diagnosis_available":true,"feedback_available":true,"method":"F1","model_calls":2,"review_required":true,"wall_seconds":75.591},{"arithmetic_status":"arithmetic_evaluated","automatic_release":false,"case_id":"eval_054","diagnosis_available":true,"feedback_available":true,"method":"F1","model_calls":2,"review_required":true,"wall_seconds":81.131},{"arithmetic_status":"rejected","automatic_release":false,"case_id":"eval_055","diagnosis_available":true,"feedback_available":true,"method":"F1","model_calls":2,"review_required":true,"wall_seconds":109.784},{"arithmetic_status":"rejected","automatic_release":false,"case_id":"eval_056","diagnosis_available":true,"feedback_available":true,"method":"F1","model_calls":2,"review_required":true,"wall_seconds":80.416},{"arithmetic_status":"arithmetic_evaluated","automatic_release":false,"case_id":"eval_057","diagnosis_available":true,"feedback_available":true,"method":"F1","model_calls":2,"review_required":true,"wall_seconds":117.015},{"arithmetic_status":"rejected","automatic_release":false,"case_id":"eval_058","diagnosis_available":true,"feedback_available":true,"method":"F1","model_calls":2,"review_required":true,"wall_seconds":89.66},{"arithmetic_status":"arithmetic_evaluated","automatic_release":false,"case_id":"eval_059","diagnosis_available":true,"feedback_available":true,"method":"F1","model_calls":2,"review_required":true,"wall_seconds":101.162},{"arithmetic_status":"rejected","automatic_release":false,"case_id":"eval_060","diagnosis_available":true,"feedback_available":true,"method":"F1","model_calls":2,"review_required":true,"wall_seconds":81.271},{"arithmetic_status":"arithmetic_evaluated","automatic_release":false,"case_id":"eval_061","diagnosis_available":true,"feedback_available":true,"method":"F1","model_calls":2,"review_required":true,"wall_seconds":63.439},{"arithmetic_status":"rejected","automatic_release":false,"case_id":"eval_062","diagnosis_available":true,"feedback_available":true,"method":"F1","model_calls":2,"review_required":true,"wall_seconds":122.421},{"arithmetic_status":"rejected","automatic_release":false,"case_id":"eval_063","diagnosis_available":true,"feedback_available":true,"method":"F1","model_calls":2,"review_required":true,"wall_seconds":82.053},{"arithmetic_status":"arithmetic_evaluated","automatic_release":false,"case_id":"eval_064","diagnosis_available":true,"feedback_available":true,"method":"F1","model_calls":2,"review_required":true,"wall_seconds":93.022},{"arithmetic_status":"arithmetic_evaluated","automatic_release":false,"case_id":"eval_065","diagnosis_available":true,"feedback_available":true,"method":"F1","model_calls":2,"review_required":true,"wall_seconds":107.439},{"arithmetic_status":"rejected","automatic_release":false,"case_id":"eval_066","diagnosis_available":true,"feedback_available":true,"method":"F1","model_calls":2,"review_required":true,"wall_seconds":69.315},{"arithmetic_status":"rejected","automatic_release":false,"case_id":"eval_067","diagnosis_available":true,"feedback_available":true,"method":"F1","model_calls":2,"review_required":true,"wall_seconds":89.876},{"arithmetic_status":"rejected","automatic_release":false,"case_id":"eval_068","diagnosis_available":true,"feedback_available":true,"method":"F1","model_calls":2,"review_required":true,"wall_seconds":130.212},{"arithmetic_status":"rejected","automatic_release":false,"case_id":"eval_069","diagnosis_available":true,"feedback_available":true,"method":"F1","model_calls":2,"review_required":true,"wall_seconds":82.877},{"arithmetic_status":"rejected","automatic_release":false,"case_id":"eval_070","diagnosis_available":true,"feedback_available":true,"method":"F1","model_calls":2,"review_required":true,"wall_seconds":82.041},{"arithmetic_status":"arithmetic_evaluated","automatic_release":false,"case_id":"eval_071","diagnosis_available":true,"feedback_available":true,"method":"F1","model_calls":2,"review_required":true,"wall_seconds":80.333},{"arithmetic_status":"rejected","automatic_release":false,"case_id":"eval_072","diagnosis_available":true,"feedback_available":true,"method":"F1","model_calls":2,"review_required":true,"wall_seconds":84.885},{"arithmetic_status":"not_attempted","automatic_release":false,"case_id":"eval_073","diagnosis_available":false,"feedback_available":false,"method":"F1","model_calls":1,"review_required":true,"wall_seconds":38.735},{"arithmetic_status":"rejected","automatic_release":false,"case_id":"eval_074","diagnosis_available":true,"feedback_available":true,"method":"F1","model_calls":2,"review_required":true,"wall_seconds":112.543},{"arithmetic_status":"rejected","automatic_release":false,"case_id":"eval_075","diagnosis_available":true,"feedback_available":true,"method":"F1","model_calls":2,"review_required":true,"wall_seconds":88.962},{"arithmetic_status":"not_proposed","automatic_release":false,"case_id":"eval_000","diagnosis_available":true,"feedback_available":true,"method":"F2.1","model_calls":2,"review_required":true,"wall_seconds":108.66},{"arithmetic_status":"not_proposed","automatic_release":false,"case_id":"eval_001","diagnosis_available":true,"feedback_available":true,"method":"F2.1","model_calls":2,"review_required":true,"wall_seconds":86.948},{"arithmetic_status":"not_proposed","automatic_release":false,"case_id":"eval_002","diagnosis_available":true,"feedback_available":true,"method":"F2.1","model_calls":2,"review_required":true,"wall_seconds":102.115},{"arithmetic_status":"not_proposed","automatic_release":false,"case_id":"eval_003","diagnosis_available":true,"feedback_available":true,"method":"F2.1","model_calls":2,"review_required":true,"wall_seconds":91.844},{"arithmetic_status":"not_proposed","automatic_release":false,"case_id":"eval_004","diagnosis_available":true,"feedback_available":true,"method":"F2.1","model_calls":2,"review_required":true,"wall_seconds":74.92},{"arithmetic_status":"not_proposed","automatic_release":false,"case_id":"eval_005","diagnosis_available":true,"feedback_available":true,"method":"F2.1","model_calls":2,"review_required":true,"wall_seconds":57.405},{"arithmetic_status":"not_proposed","automatic_release":false,"case_id":"eval_006","diagnosis_available":true,"feedback_available":true,"method":"F2.1","model_calls":2,"review_required":true,"wall_seconds":94.359},{"arithmetic_status":"not_proposed","automatic_release":false,"case_id":"eval_007","diagnosis_available":true,"feedback_available":true,"method":"F2.1","model_calls":2,"review_required":true,"wall_seconds":84.378},{"arithmetic_status":"not_proposed","automatic_release":false,"case_id":"eval_008","diagnosis_available":true,"feedback_available":true,"method":"F2.1","model_calls":2,"review_required":true,"wall_seconds":148.132},{"arithmetic_status":"not_proposed","automatic_release":false,"case_id":"eval_009","diagnosis_available":true,"feedback_available":true,"method":"F2.1","model_calls":2,"review_required":true,"wall_seconds":91.573},{"arithmetic_status":"not_proposed","automatic_release":false,"case_id":"eval_010","diagnosis_available":true,"feedback_available":true,"method":"F2.1","model_calls":2,"review_required":true,"wall_seconds":93.224},{"arithmetic_status":"not_proposed","automatic_release":false,"case_id":"eval_011","diagnosis_available":true,"feedback_available":true,"method":"F2.1","model_calls":2,"review_required":true,"wall_seconds":107.357},{"arithmetic_status":"not_proposed","automatic_release":false,"case_id":"eval_012","diagnosis_available":true,"feedback_available":true,"method":"F2.1","model_calls":2,"review_required":true,"wall_seconds":82.573},{"arithmetic_status":"not_proposed","automatic_release":false,"case_id":"eval_013","diagnosis_available":true,"feedback_available":true,"method":"F2.1","model_calls":2,"review_required":true,"wall_seconds":106.403},{"arithmetic_status":"not_proposed","automatic_release":false,"case_id":"eval_014","diagnosis_available":true,"feedback_available":true,"method":"F2.1","model_calls":2,"review_required":true,"wall_seconds":104.668},{"arithmetic_status":"not_proposed","automatic_release":false,"case_id":"eval_015","diagnosis_available":true,"feedback_available":true,"method":"F2.1","model_calls":2,"review_required":true,"wall_seconds":117.282},{"arithmetic_status":"not_proposed","automatic_release":false,"case_id":"eval_016","diagnosis_available":true,"feedback_available":true,"method":"F2.1","model_calls":2,"review_required":true,"wall_seconds":105.653},{"arithmetic_status":"not_proposed","automatic_release":false,"case_id":"eval_017","diagnosis_available":true,"feedback_available":true,"method":"F2.1","model_calls":2,"review_required":true,"wall_seconds":95.799},{"arithmetic_status":"not_proposed","automatic_release":false,"case_id":"eval_018","diagnosis_available":true,"feedback_available":true,"method":"F2.1","model_calls":2,"review_required":true,"wall_seconds":82.385},{"arithmetic_status":"not_proposed","automatic_release":false,"case_id":"eval_019","diagnosis_available":true,"feedback_available":true,"method":"F2.1","model_calls":2,"review_required":true,"wall_seconds":86.175},{"arithmetic_status":"not_proposed","automatic_release":false,"case_id":"eval_020","diagnosis_available":true,"feedback_available":true,"method":"F2.1","model_calls":2,"review_required":true,"wall_seconds":88.615},{"arithmetic_status":"not_proposed","automatic_release":false,"case_id":"eval_021","diagnosis_available":true,"feedback_available":true,"method":"F2.1","model_calls":2,"review_required":true,"wall_seconds":91.53},{"arithmetic_status":"not_proposed","automatic_release":false,"case_id":"eval_022","diagnosis_available":true,"feedback_available":true,"method":"F2.1","model_calls":2,"review_required":true,"wall_seconds":103.199},{"arithmetic_status":"not_proposed","automatic_release":false,"case_id":"eval_023","diagnosis_available":true,"feedback_available":true,"method":"F2.1","model_calls":2,"review_required":true,"wall_seconds":93.986},{"arithmetic_status":"not_proposed","automatic_release":false,"case_id":"eval_024","diagnosis_available":true,"feedback_available":true,"method":"F2.1","model_calls":2,"review_required":true,"wall_seconds":116.412},{"arithmetic_status":"not_proposed","automatic_release":false,"case_id":"eval_025","diagnosis_available":true,"feedback_available":true,"method":"F2.1","model_calls":2,"review_required":true,"wall_seconds":79.536},{"arithmetic_status":"not_proposed","automatic_release":false,"case_id":"eval_026","diagnosis_available":true,"feedback_available":true,"method":"F2.1","model_calls":2,"review_required":true,"wall_seconds":79.666},{"arithmetic_status":"not_proposed","automatic_release":false,"case_id":"eval_027","diagnosis_available":true,"feedback_available":true,"method":"F2.1","model_calls":2,"review_required":true,"wall_seconds":70.207},{"arithmetic_status":"not_proposed","automatic_release":false,"case_id":"eval_028","diagnosis_available":true,"feedback_available":true,"method":"F2.1","model_calls":2,"review_required":true,"wall_seconds":97.172},{"arithmetic_status":"not_proposed","automatic_release":false,"case_id":"eval_029","diagnosis_available":true,"feedback_available":true,"method":"F2.1","model_calls":2,"review_required":true,"wall_seconds":134.268},{"arithmetic_status":"not_proposed","automatic_release":false,"case_id":"eval_030","diagnosis_available":true,"feedback_available":true,"method":"F2.1","model_calls":2,"review_required":true,"wall_seconds":127.046},{"arithmetic_status":"not_proposed","automatic_release":false,"case_id":"eval_031","diagnosis_available":true,"feedback_available":true,"method":"F2.1","model_calls":2,"review_required":true,"wall_seconds":93.304},{"arithmetic_status":"not_proposed","automatic_release":false,"case_id":"eval_032","diagnosis_available":true,"feedback_available":true,"method":"F2.1","model_calls":2,"review_required":true,"wall_seconds":159.56},{"arithmetic_status":"not_proposed","automatic_release":false,"case_id":"eval_033","diagnosis_available":true,"feedback_available":true,"method":"F2.1","model_calls":2,"review_required":true,"wall_seconds":100.876},{"arithmetic_status":"not_proposed","automatic_release":false,"case_id":"eval_034","diagnosis_available":true,"feedback_available":true,"method":"F2.1","model_calls":2,"review_required":true,"wall_seconds":88.939},{"arithmetic_status":"not_proposed","automatic_release":false,"case_id":"eval_035","diagnosis_available":true,"feedback_available":true,"method":"F2.1","model_calls":2,"review_required":true,"wall_seconds":115.558},{"arithmetic_status":"not_proposed","automatic_release":false,"case_id":"eval_036","diagnosis_available":true,"feedback_available":true,"method":"F2.1","model_calls":2,"review_required":true,"wall_seconds":90.633},{"arithmetic_status":"not_proposed","automatic_release":false,"case_id":"eval_037","diagnosis_available":true,"feedback_available":true,"method":"F2.1","model_calls":2,"review_required":true,"wall_seconds":102.939},{"arithmetic_status":"not_proposed","automatic_release":false,"case_id":"eval_038","diagnosis_available":true,"feedback_available":true,"method":"F2.1","model_calls":2,"review_required":true,"wall_seconds":102.257},{"arithmetic_status":"not_proposed","automatic_release":false,"case_id":"eval_039","diagnosis_available":true,"feedback_available":true,"method":"F2.1","model_calls":2,"review_required":true,"wall_seconds":101.916},{"arithmetic_status":"not_proposed","automatic_release":false,"case_id":"eval_040","diagnosis_available":true,"feedback_available":true,"method":"F2.1","model_calls":2,"review_required":true,"wall_seconds":91.387},{"arithmetic_status":"not_proposed","automatic_release":false,"case_id":"eval_041","diagnosis_available":true,"feedback_available":true,"method":"F2.1","model_calls":2,"review_required":true,"wall_seconds":97.837},{"arithmetic_status":"not_proposed","automatic_release":false,"case_id":"eval_042","diagnosis_available":true,"feedback_available":true,"method":"F2.1","model_calls":2,"review_required":true,"wall_seconds":105.065},{"arithmetic_status":"arithmetic_evaluated","automatic_release":false,"case_id":"eval_043","diagnosis_available":true,"feedback_available":true,"method":"F2.1","model_calls":2,"review_required":true,"wall_seconds":91.476},{"arithmetic_status":"not_proposed","automatic_release":false,"case_id":"eval_044","diagnosis_available":true,"feedback_available":true,"method":"F2.1","model_calls":2,"review_required":true,"wall_seconds":101.04},{"arithmetic_status":"not_proposed","automatic_release":false,"case_id":"eval_045","diagnosis_available":true,"feedback_available":true,"method":"F2.1","model_calls":2,"review_required":true,"wall_seconds":116.026},{"arithmetic_status":"not_proposed","automatic_release":false,"case_id":"eval_046","diagnosis_available":true,"feedback_available":true,"method":"F2.1","model_calls":2,"review_required":true,"wall_seconds":101.528},{"arithmetic_status":"not_proposed","automatic_release":false,"case_id":"eval_047","diagnosis_available":true,"feedback_available":true,"method":"F2.1","model_calls":2,"review_required":true,"wall_seconds":115.022},{"arithmetic_status":"not_proposed","automatic_release":false,"case_id":"eval_048","diagnosis_available":true,"feedback_available":true,"method":"F2.1","model_calls":2,"review_required":true,"wall_seconds":124.754},{"arithmetic_status":"not_proposed","automatic_release":false,"case_id":"eval_049","diagnosis_available":true,"feedback_available":true,"method":"F2.1","model_calls":2,"review_required":true,"wall_seconds":108.413},{"arithmetic_status":"not_proposed","automatic_release":false,"case_id":"eval_050","diagnosis_available":true,"feedback_available":true,"method":"F2.1","model_calls":2,"review_required":true,"wall_seconds":97.679},{"arithmetic_status":"not_proposed","automatic_release":false,"case_id":"eval_051","diagnosis_available":true,"feedback_available":true,"method":"F2.1","model_calls":2,"review_required":true,"wall_seconds":77.356},{"arithmetic_status":"not_proposed","automatic_release":false,"case_id":"eval_052","diagnosis_available":true,"feedback_available":true,"method":"F2.1","model_calls":2,"review_required":true,"wall_seconds":126.399},{"arithmetic_status":"arithmetic_evaluated","automatic_release":false,"case_id":"eval_053","diagnosis_available":true,"feedback_available":true,"method":"F2.1","model_calls":2,"review_required":true,"wall_seconds":98.432},{"arithmetic_status":"not_proposed","automatic_release":false,"case_id":"eval_054","diagnosis_available":true,"feedback_available":true,"method":"F2.1","model_calls":2,"review_required":true,"wall_seconds":123.22},{"arithmetic_status":"not_proposed","automatic_release":false,"case_id":"eval_055","diagnosis_available":true,"feedback_available":true,"method":"F2.1","model_calls":2,"review_required":true,"wall_seconds":109.774},{"arithmetic_status":"not_proposed","automatic_release":false,"case_id":"eval_056","diagnosis_available":true,"feedback_available":true,"method":"F2.1","model_calls":2,"review_required":true,"wall_seconds":107.012},{"arithmetic_status":"not_proposed","automatic_release":false,"case_id":"eval_057","diagnosis_available":true,"feedback_available":true,"method":"F2.1","model_calls":2,"review_required":true,"wall_seconds":98.792},{"arithmetic_status":"not_proposed","automatic_release":false,"case_id":"eval_058","diagnosis_available":true,"feedback_available":true,"method":"F2.1","model_calls":2,"review_required":true,"wall_seconds":82.133},{"arithmetic_status":"not_proposed","automatic_release":false,"case_id":"eval_059","diagnosis_available":true,"feedback_available":true,"method":"F2.1","model_calls":2,"review_required":true,"wall_seconds":100.853},{"arithmetic_status":"not_proposed","automatic_release":false,"case_id":"eval_060","diagnosis_available":true,"feedback_available":true,"method":"F2.1","model_calls":2,"review_required":true,"wall_seconds":98.552},{"arithmetic_status":"not_proposed","automatic_release":false,"case_id":"eval_061","diagnosis_available":true,"feedback_available":true,"method":"F2.1","model_calls":2,"review_required":true,"wall_seconds":166.943},{"arithmetic_status":"not_proposed","automatic_release":false,"case_id":"eval_062","diagnosis_available":true,"feedback_available":true,"method":"F2.1","model_calls":2,"review_required":true,"wall_seconds":144.709},{"arithmetic_status":"not_proposed","automatic_release":false,"case_id":"eval_063","diagnosis_available":true,"feedback_available":true,"method":"F2.1","model_calls":2,"review_required":true,"wall_seconds":95.184},{"arithmetic_status":"not_proposed","automatic_release":false,"case_id":"eval_064","diagnosis_available":true,"feedback_available":true,"method":"F2.1","model_calls":2,"review_required":true,"wall_seconds":98.451},{"arithmetic_status":"not_proposed","automatic_release":false,"case_id":"eval_065","diagnosis_available":true,"feedback_available":true,"method":"F2.1","model_calls":2,"review_required":true,"wall_seconds":111.983},{"arithmetic_status":"not_proposed","automatic_release":false,"case_id":"eval_066","diagnosis_available":true,"feedback_available":true,"method":"F2.1","model_calls":2,"review_required":true,"wall_seconds":110.27},{"arithmetic_status":"not_proposed","automatic_release":false,"case_id":"eval_067","diagnosis_available":true,"feedback_available":true,"method":"F2.1","model_calls":2,"review_required":true,"wall_seconds":94.179},{"arithmetic_status":"not_proposed","automatic_release":false,"case_id":"eval_068","diagnosis_available":true,"feedback_available":true,"method":"F2.1","model_calls":2,"review_required":true,"wall_seconds":128.477},{"arithmetic_status":"not_proposed","automatic_release":false,"case_id":"eval_069","diagnosis_available":true,"feedback_available":true,"method":"F2.1","model_calls":2,"review_required":true,"wall_seconds":119.936},{"arithmetic_status":"not_proposed","automatic_release":false,"case_id":"eval_070","diagnosis_available":true,"feedback_available":true,"method":"F2.1","model_calls":2,"review_required":true,"wall_seconds":95.488},{"arithmetic_status":"not_proposed","automatic_release":false,"case_id":"eval_071","diagnosis_available":true,"feedback_available":true,"method":"F2.1","model_calls":2,"review_required":true,"wall_seconds":114.449},{"arithmetic_status":"not_proposed","automatic_release":false,"case_id":"eval_072","diagnosis_available":true,"feedback_available":true,"method":"F2.1","model_calls":2,"review_required":true,"wall_seconds":111.157},{"arithmetic_status":"not_proposed","automatic_release":false,"case_id":"eval_073","diagnosis_available":true,"feedback_available":true,"method":"F2.1","model_calls":2,"review_required":true,"wall_seconds":126.356},{"arithmetic_status":"not_proposed","automatic_release":false,"case_id":"eval_074","diagnosis_available":true,"feedback_available":true,"method":"F2.1","model_calls":2,"review_required":true,"wall_seconds":149.161},{"arithmetic_status":"not_proposed","automatic_release":false,"case_id":"eval_075","diagnosis_available":true,"feedback_available":true,"method":"F2.1","model_calls":2,"review_required":true,"wall_seconds":130.701}],"expert":[{"available":true,"critical_error":1,"diagnosis":1,"evidence":1,"mathematics":0,"method":"F2.1","pedagogy":0,"problem_code":"P30"},{"available":true,"critical_error":0,"diagnosis":2,"evidence":2,"mathematics":2,"method":"F2.1","pedagogy":1,"problem_code":"P24"},{"available":true,"critical_error":0,"diagnosis":1,"evidence":1,"mathematics":1,"method":"F2.1","pedagogy":1,"problem_code":"P02"},{"available":true,"critical_error":1,"diagnosis":1,"evidence":1,"mathematics":0,"method":"F1","pedagogy":0,"problem_code":"P10"},{"available":true,"critical_error":0,"diagnosis":2,"evidence":2,"mathematics":2,"method":"F1","pedagogy":1,"problem_code":"P21"},{"available":true,"critical_error":0,"diagnosis":2,"evidence":2,"mathematics":2,"method":"F1","pedagogy":1,"problem_code":"P09"},{"available":true,"critical_error":0,"diagnosis":0,"evidence":1,"mathematics":1,"method":"F2.1","pedagogy":1,"problem_code":"P18"},{"available":true,"critical_error":1,"diagnosis":0,"evidence":1,"mathematics":0,"method":"F1","pedagogy":0,"problem_code":"P13"},{"available":true,"critical_error":1,"diagnosis":1,"evidence":1,"mathematics":0,"method":"F1","pedagogy":0,"problem_code":"P19"},{"available":true,"critical_error":0,"diagnosis":1,"evidence":1,"mathematics":1,"method":"F1","pedagogy":0,"problem_code":"P08"},{"available":true,"critical_error":0,"diagnosis":2,"evidence":2,"mathematics":1,"method":"F1","pedagogy":1,"problem_code":"P16"},{"available":true,"critical_error":0,"diagnosis":2,"evidence":2,"mathematics":1,"method":"F1","pedagogy":1,"problem_code":"P30"},{"available":true,"critical_error":1,"diagnosis":1,"evidence":1,"mathematics":0,"method":"F2.1","pedagogy":1,"problem_code":"P04"},{"available":true,"critical_error":0,"diagnosis":1,"evidence":1,"mathematics":0,"method":"F2.1","pedagogy":1,"problem_code":"P16"},{"available":true,"critical_error":1,"diagnosis":0,"evidence":0,"mathematics":0,"method":"F2.1","pedagogy":0,"problem_code":"P08"},{"available":true,"critical_error":0,"diagnosis":2,"evidence":2,"mathematics":2,"method":"F2.1","pedagogy":2,"problem_code":"P01"},{"available":true,"critical_error":1,"diagnosis":1,"evidence":1,"mathematics":0,"method":"F1","pedagogy":0,"problem_code":"P11"},{"available":false,"critical_error":null,"diagnosis":null,"evidence":null,"mathematics":null,"method":"F1","pedagogy":null,"problem_code":"P20"},{"available":true,"critical_error":0,"diagnosis":2,"evidence":2,"mathematics":2,"method":"F1","pedagogy":1,"problem_code":"P28"},{"available":true,"critical_error":1,"diagnosis":0,"evidence":0,"mathematics":0,"method":"F2.1","pedagogy":0,"problem_code":"P17"},{"available":true,"critical_error":0,"diagnosis":2,"evidence":2,"mathematics":2,"method":"F2.1","pedagogy":2,"problem_code":"P21"},{"available":true,"critical_error":0,"diagnosis":1,"evidence":1,"mathematics":1,"method":"F2.1","pedagogy":1,"problem_code":"P19"},{"available":true,"critical_error":1,"diagnosis":1,"evidence":1,"mathematics":0,"method":"F1","pedagogy":0,"problem_code":"P05"},{"available":true,"critical_error":1,"diagnosis":0,"evidence":0,"mathematics":0,"method":"F2.1","pedagogy":0,"problem_code":"P14"},{"available":true,"critical_error":1,"diagnosis":0,"evidence":0,"mathematics":0,"method":"F2.1","pedagogy":0,"problem_code":"P20"},{"available":true,"critical_error":0,"diagnosis":2,"evidence":2,"mathematics":2,"method":"F2.1","pedagogy":2,"problem_code":"P28"},{"available":true,"critical_error":1,"diagnosis":1,"evidence":1,"mathematics":0,"method":"F2.1","pedagogy":1,"problem_code":"P12"},{"available":true,"critical_error":1,"diagnosis":0,"evidence":0,"mathematics":0,"method":"F2.1","pedagogy":1,"problem_code":"P29"},{"available":true,"critical_error":1,"diagnosis":0,"evidence":0,"mathematics":0,"method":"F2.1","pedagogy":0,"problem_code":"P15"},{"available":true,"critical_error":1,"diagnosis":0,"evidence":1,"mathematics":1,"method":"F1","pedagogy":0,"problem_code":"P23"},{"available":false,"critical_error":null,"diagnosis":null,"evidence":null,"mathematics":null,"method":"F1","pedagogy":null,"problem_code":"P26"},{"available":true,"critical_error":1,"diagnosis":0,"evidence":0,"mathematics":0,"method":"F1","pedagogy":0,"problem_code":"P18"},{"available":true,"critical_error":0,"diagnosis":1,"evidence":1,"mathematics":1,"method":"F2.1","pedagogy":1,"problem_code":"P11"},{"available":true,"critical_error":1,"diagnosis":1,"evidence":1,"mathematics":0,"method":"F2.1","pedagogy":0,"problem_code":"P10"},{"available":true,"critical_error":0,"diagnosis":2,"evidence":2,"mathematics":2,"method":"F1","pedagogy":1,"problem_code":"P01"},{"available":true,"critical_error":0,"diagnosis":1,"evidence":1,"mathematics":1,"method":"F2.1","pedagogy":1,"problem_code":"P22"},{"available":true,"critical_error":1,"diagnosis":0,"evidence":0,"mathematics":0,"method":"F1","pedagogy":1,"problem_code":"P25"},{"available":true,"critical_error":0,"diagnosis":2,"evidence":2,"mathematics":1,"method":"F2.1","pedagogy":2,"problem_code":"P09"},{"available":true,"critical_error":0,"diagnosis":2,"evidence":2,"mathematics":2,"method":"F1","pedagogy":1,"problem_code":"P07"},{"available":true,"critical_error":1,"diagnosis":0,"evidence":0,"mathematics":0,"method":"F1","pedagogy":0,"problem_code":"P22"},{"available":true,"critical_error":1,"diagnosis":0,"evidence":0,"mathematics":0,"method":"F1","pedagogy":0,"problem_code":"P29"},{"available":true,"critical_error":1,"diagnosis":1,"evidence":1,"mathematics":0,"method":"F2.1","pedagogy":1,"problem_code":"P27"},{"available":true,"critical_error":1,"diagnosis":0,"evidence":0,"mathematics":0,"method":"F1","pedagogy":0,"problem_code":"P14"},{"available":true,"critical_error":0,"diagnosis":2,"evidence":2,"mathematics":2,"method":"F2.1","pedagogy":1,"problem_code":"P06"},{"available":true,"critical_error":0,"diagnosis":0,"evidence":1,"mathematics":1,"method":"F1","pedagogy":0,"problem_code":"P02"},{"available":true,"critical_error":1,"diagnosis":0,"evidence":0,"mathematics":0,"method":"F1","pedagogy":0,"problem_code":"P24"},{"available":true,"critical_error":0,"diagnosis":2,"evidence":2,"mathematics":2,"method":"F1","pedagogy":1,"problem_code":"P17"},{"available":true,"critical_error":0,"diagnosis":2,"evidence":2,"mathematics":2,"method":"F1","pedagogy":1,"problem_code":"P04"},{"available":true,"critical_error":0,"diagnosis":0,"evidence":1,"mathematics":1,"method":"F2.1","pedagogy":1,"problem_code":"P07"},{"available":true,"critical_error":0,"diagnosis":1,"evidence":1,"mathematics":1,"method":"F2.1","pedagogy":1,"problem_code":"P05"},{"available":true,"critical_error":0,"diagnosis":2,"evidence":2,"mathematics":2,"method":"F1","pedagogy":1,"problem_code":"P27"},{"available":true,"critical_error":0,"diagnosis":1,"evidence":1,"mathematics":1,"method":"F2.1","pedagogy":1,"problem_code":"P25"},{"available":true,"critical_error":0,"diagnosis":2,"evidence":2,"mathematics":2,"method":"F1","pedagogy":1,"problem_code":"P06"},{"available":true,"critical_error":0,"diagnosis":2,"evidence":2,"mathematics":2,"method":"F2.1","pedagogy":2,"problem_code":"P03"},{"available":true,"critical_error":1,"diagnosis":0,"evidence":0,"mathematics":0,"method":"F1","pedagogy":0,"problem_code":"P15"},{"available":true,"critical_error":0,"diagnosis":1,"evidence":1,"mathematics":1,"method":"F2.1","pedagogy":1,"problem_code":"P26"},{"available":true,"critical_error":1,"diagnosis":0,"evidence":1,"mathematics":0,"method":"F2.1","pedagogy":0,"problem_code":"P13"},{"available":true,"critical_error":0,"diagnosis":2,"evidence":2,"mathematics":2,"method":"F1","pedagogy":2,"problem_code":"P12"},{"available":true,"critical_error":1,"diagnosis":0,"evidence":0,"mathematics":0,"method":"F1","pedagogy":0,"problem_code":"P03"},{"available":true,"critical_error":0,"diagnosis":2,"evidence":2,"mathematics":2,"method":"F2.1","pedagogy":2,"problem_code":"P23"}],"development":[{"case_id":"dev_004","critical_error":0,"diagnosis":2,"evidence":2,"mathematics":2,"method":"F1","pedagogy":1,"reviewer":"R1"},{"case_id":"dev_015","critical_error":0,"diagnosis":2,"evidence":2,"mathematics":2,"method":"F2.1","pedagogy":2,"reviewer":"R1"},{"case_id":"dev_015","critical_error":1,"diagnosis":1,"evidence":1,"mathematics":1,"method":"F1","pedagogy":1,"reviewer":"R1"},{"case_id":"dev_003","critical_error":0,"diagnosis":1,"evidence":1,"mathematics":1,"method":"F2.2","pedagogy":1,"reviewer":"R1"},{"case_id":"dev_024","critical_error":1,"diagnosis":0,"evidence":0,"mathematics":0,"method":"F2.1","pedagogy":0,"reviewer":"R1"},{"case_id":"dev_003","critical_error":1,"diagnosis":2,"evidence":2,"mathematics":1,"method":"F1","pedagogy":2,"reviewer":"R1"},{"case_id":"dev_016","critical_error":0,"diagnosis":1,"evidence":2,"mathematics":1,"method":"F1","pedagogy":1,"reviewer":"R1"},{"case_id":"dev_024","critical_error":0,"diagnosis":2,"evidence":2,"mathematics":2,"method":"F2.2","pedagogy":1,"reviewer":"R1"},{"case_id":"dev_007","critical_error":0,"diagnosis":1,"evidence":1,"mathematics":1,"method":"F1","pedagogy":1,"reviewer":"R1"},{"case_id":"dev_007","critical_error":0,"diagnosis":2,"evidence":1,"mathematics":1,"method":"F2.1","pedagogy":2,"reviewer":"R1"},{"case_id":"dev_019","critical_error":1,"diagnosis":0,"evidence":0,"mathematics":1,"method":"F2.2","pedagogy":0,"reviewer":"R1"},{"case_id":"dev_012","critical_error":0,"diagnosis":2,"evidence":2,"mathematics":1,"method":"F2.2","pedagogy":2,"reviewer":"R1"},{"case_id":"dev_019","critical_error":0,"diagnosis":2,"evidence":2,"mathematics":2,"method":"F2.1","pedagogy":1,"reviewer":"R1"},{"case_id":"dev_015","critical_error":0,"diagnosis":1,"evidence":1,"mathematics":1,"method":"F2.2","pedagogy":0,"reviewer":"R1"},{"case_id":"dev_011","critical_error":0,"diagnosis":1,"evidence":2,"mathematics":1,"method":"F2.1","pedagogy":2,"reviewer":"R1"},{"case_id":"dev_014","critical_error":1,"diagnosis":1,"evidence":0,"mathematics":0,"method":"F2.2","pedagogy":1,"reviewer":"R1"},{"case_id":"dev_016","critical_error":1,"diagnosis":0,"evidence":0,"mathematics":1,"method":"F2.2","pedagogy":0,"reviewer":"R1"},{"case_id":"dev_027","critical_error":0,"diagnosis":1,"evidence":1,"mathematics":1,"method":"F2.2","pedagogy":1,"reviewer":"R1"},{"case_id":"dev_004","critical_error":0,"diagnosis":2,"evidence":2,"mathematics":2,"method":"F2.1","pedagogy":2,"reviewer":"R1"},{"case_id":"dev_019","critical_error":0,"diagnosis":2,"evidence":2,"mathematics":2,"method":"F1","pedagogy":1,"reviewer":"R1"},{"case_id":"dev_020","critical_error":0,"diagnosis":2,"evidence":2,"mathematics":2,"method":"F2.1","pedagogy":1,"reviewer":"R1"},{"case_id":"dev_014","critical_error":0,"diagnosis":2,"evidence":2,"mathematics":2,"method":"F1","pedagogy":1,"reviewer":"R1"},{"case_id":"dev_027","critical_error":0,"diagnosis":2,"evidence":2,"mathematics":1,"method":"F2.1","pedagogy":2,"reviewer":"R1"},{"case_id":"dev_024","critical_error":1,"diagnosis":0,"evidence":0,"mathematics":0,"method":"F1","pedagogy":0,"reviewer":"R1"},{"case_id":"dev_020","critical_error":0,"diagnosis":2,"evidence":2,"mathematics":2,"method":"F2.2","pedagogy":1,"reviewer":"R1"},{"case_id":"dev_011","critical_error":0,"diagnosis":2,"evidence":2,"mathematics":2,"method":"F2.2","pedagogy":2,"reviewer":"R1"},{"case_id":"dev_014","critical_error":0,"diagnosis":2,"evidence":2,"mathematics":1,"method":"F2.1","pedagogy":0,"reviewer":"R1"},{"case_id":"dev_027","critical_error":0,"diagnosis":1,"evidence":1,"mathematics":1,"method":"F1","pedagogy":1,"reviewer":"R1"},{"case_id":"dev_004","critical_error":0,"diagnosis":2,"evidence":1,"mathematics":1,"method":"F2.2","pedagogy":1,"reviewer":"R1"},{"case_id":"dev_007","critical_error":1,"diagnosis":1,"evidence":1,"mathematics":0,"method":"F2.2","pedagogy":1,"reviewer":"R1"},{"case_id":"dev_003","critical_error":0,"diagnosis":1,"evidence":1,"mathematics":1,"method":"F2.1","pedagogy":2,"reviewer":"R1"},{"case_id":"dev_011","critical_error":1,"diagnosis":0,"evidence":0,"mathematics":0,"method":"F1","pedagogy":0,"reviewer":"R1"},{"case_id":"dev_016","critical_error":0,"diagnosis":1,"evidence":1,"mathematics":1,"method":"F2.1","pedagogy":1,"reviewer":"R1"},{"case_id":"dev_020","critical_error":0,"diagnosis":1,"evidence":2,"mathematics":1,"method":"F1","pedagogy":1,"reviewer":"R1"},{"case_id":"dev_012","critical_error":0,"diagnosis":1,"evidence":0,"mathematics":1,"method":"F2.1","pedagogy":1,"reviewer":"R1"},{"case_id":"dev_004","critical_error":0,"diagnosis":2,"evidence":2,"mathematics":2,"method":"F1","pedagogy":1,"reviewer":"R2"},{"case_id":"dev_015","critical_error":0,"diagnosis":2,"evidence":2,"mathematics":2,"method":"F2.1","pedagogy":2,"reviewer":"R2"},{"case_id":"dev_015","critical_error":1,"diagnosis":1,"evidence":1,"mathematics":1,"method":"F1","pedagogy":1,"reviewer":"R2"},{"case_id":"dev_003","critical_error":0,"diagnosis":1,"evidence":1,"mathematics":1,"method":"F2.2","pedagogy":1,"reviewer":"R2"},{"case_id":"dev_024","critical_error":1,"diagnosis":0,"evidence":0,"mathematics":0,"method":"F2.1","pedagogy":0,"reviewer":"R2"},{"case_id":"dev_003","critical_error":1,"diagnosis":2,"evidence":2,"mathematics":1,"method":"F1","pedagogy":2,"reviewer":"R2"},{"case_id":"dev_016","critical_error":0,"diagnosis":1,"evidence":2,"mathematics":1,"method":"F1","pedagogy":1,"reviewer":"R2"},{"case_id":"dev_024","critical_error":0,"diagnosis":2,"evidence":2,"mathematics":1,"method":"F2.2","pedagogy":1,"reviewer":"R2"},{"case_id":"dev_007","critical_error":0,"diagnosis":1,"evidence":1,"mathematics":1,"method":"F1","pedagogy":1,"reviewer":"R2"},{"case_id":"dev_007","critical_error":0,"diagnosis":2,"evidence":1,"mathematics":1,"method":"F2.1","pedagogy":2,"reviewer":"R2"},{"case_id":"dev_019","critical_error":1,"diagnosis":0,"evidence":0,"mathematics":0,"method":"F2.2","pedagogy":0,"reviewer":"R2"},{"case_id":"dev_012","critical_error":0,"diagnosis":2,"evidence":2,"mathematics":1,"method":"F2.2","pedagogy":1,"reviewer":"R2"},{"case_id":"dev_019","critical_error":0,"diagnosis":2,"evidence":2,"mathematics":2,"method":"F2.1","pedagogy":1,"reviewer":"R2"},{"case_id":"dev_015","critical_error":0,"diagnosis":2,"evidence":2,"mathematics":1,"method":"F2.2","pedagogy":0,"reviewer":"R2"},{"case_id":"dev_011","critical_error":0,"diagnosis":1,"evidence":1,"mathematics":1,"method":"F2.1","pedagogy":2,"reviewer":"R2"},{"case_id":"dev_014","critical_error":1,"diagnosis":0,"evidence":0,"mathematics":0,"method":"F2.2","pedagogy":0,"reviewer":"R2"},{"case_id":"dev_016","critical_error":0,"diagnosis":1,"evidence":1,"mathematics":1,"method":"F2.2","pedagogy":1,"reviewer":"R2"},{"case_id":"dev_027","critical_error":0,"diagnosis":1,"evidence":1,"mathematics":1,"method":"F2.2","pedagogy":1,"reviewer":"R2"},{"case_id":"dev_004","critical_error":0,"diagnosis":2,"evidence":2,"mathematics":2,"method":"F2.1","pedagogy":2,"reviewer":"R2"},{"case_id":"dev_019","critical_error":0,"diagnosis":2,"evidence":2,"mathematics":2,"method":"F1","pedagogy":1,"reviewer":"R2"},{"case_id":"dev_020","critical_error":0,"diagnosis":2,"evidence":2,"mathematics":2,"method":"F2.1","pedagogy":1,"reviewer":"R2"},{"case_id":"dev_014","critical_error":0,"diagnosis":2,"evidence":2,"mathematics":2,"method":"F1","pedagogy":1,"reviewer":"R2"},{"case_id":"dev_027","critical_error":0,"diagnosis":1,"evidence":1,"mathematics":1,"method":"F2.1","pedagogy":1,"reviewer":"R2"},{"case_id":"dev_024","critical_error":1,"diagnosis":0,"evidence":0,"mathematics":0,"method":"F1","pedagogy":1,"reviewer":"R2"},{"case_id":"dev_020","critical_error":0,"diagnosis":2,"evidence":2,"mathematics":2,"method":"F2.2","pedagogy":1,"reviewer":"R2"},{"case_id":"dev_011","critical_error":0,"diagnosis":2,"evidence":2,"mathematics":2,"method":"F2.2","pedagogy":2,"reviewer":"R2"},{"case_id":"dev_014","critical_error":0,"diagnosis":2,"evidence":2,"mathematics":2,"method":"F2.1","pedagogy":0,"reviewer":"R2"},{"case_id":"dev_027","critical_error":1,"diagnosis":1,"evidence":1,"mathematics":0,"method":"F1","pedagogy":0,"reviewer":"R2"},{"case_id":"dev_004","critical_error":0,"diagnosis":2,"evidence":1,"mathematics":1,"method":"F2.2","pedagogy":2,"reviewer":"R2"},{"case_id":"dev_007","critical_error":1,"diagnosis":1,"evidence":1,"mathematics":0,"method":"F2.2","pedagogy":1,"reviewer":"R2"},{"case_id":"dev_003","critical_error":0,"diagnosis":1,"evidence":1,"mathematics":1,"method":"F2.1","pedagogy":2,"reviewer":"R2"},{"case_id":"dev_011","critical_error":1,"diagnosis":0,"evidence":0,"mathematics":0,"method":"F1","pedagogy":0,"reviewer":"R2"},{"case_id":"dev_016","critical_error":0,"diagnosis":1,"evidence":1,"mathematics":1,"method":"F2.1","pedagogy":1,"reviewer":"R2"},{"case_id":"dev_020","critical_error":0,"diagnosis":1,"evidence":1,"mathematics":1,"method":"F1","pedagogy":1,"reviewer":"R2"},{"case_id":"dev_012","critical_error":0,"diagnosis":2,"evidence":1,"mathematics":1,"method":"F2.1","pedagogy":2,"reviewer":"R2"},{"case_id":"dev_004","critical_error":0,"diagnosis":2,"evidence":2,"mathematics":2,"method":"F1","pedagogy":2,"reviewer":"R3"},{"case_id":"dev_015","critical_error":0,"diagnosis":2,"evidence":2,"mathematics":2,"method":"F2.1","pedagogy":2,"reviewer":"R3"},{"case_id":"dev_015","critical_error":1,"diagnosis":0,"evidence":1,"mathematics":0,"method":"F1","pedagogy":2,"reviewer":"R3"},{"case_id":"dev_003","critical_error":0,"diagnosis":2,"evidence":2,"mathematics":2,"method":"F2.2","pedagogy":2,"reviewer":"R3"},{"case_id":"dev_024","critical_error":1,"diagnosis":0,"evidence":1,"mathematics":0,"method":"F2.1","pedagogy":0,"reviewer":"R3"},{"case_id":"dev_003","critical_error":1,"diagnosis":2,"evidence":2,"mathematics":0,"method":"F1","pedagogy":2,"reviewer":"R3"},{"case_id":"dev_016","critical_error":1,"diagnosis":0,"evidence":1,"mathematics":0,"method":"F1","pedagogy":1,"reviewer":"R3"},{"case_id":"dev_024","critical_error":0,"diagnosis":2,"evidence":2,"mathematics":1,"method":"F2.2","pedagogy":2,"reviewer":"R3"},{"case_id":"dev_007","critical_error":0,"diagnosis":1,"evidence":1,"mathematics":1,"method":"F1","pedagogy":2,"reviewer":"R3"},{"case_id":"dev_007","critical_error":0,"diagnosis":1,"evidence":1,"mathematics":2,"method":"F2.1","pedagogy":2,"reviewer":"R3"},{"case_id":"dev_019","critical_error":1,"diagnosis":0,"evidence":1,"mathematics":1,"method":"F2.2","pedagogy":0,"reviewer":"R3"},{"case_id":"dev_012","critical_error":0,"diagnosis":2,"evidence":2,"mathematics":1,"method":"F2.2","pedagogy":2,"reviewer":"R3"},{"case_id":"dev_019","critical_error":0,"diagnosis":2,"evidence":2,"mathematics":2,"method":"F2.1","pedagogy":2,"reviewer":"R3"},{"case_id":"dev_015","critical_error":0,"diagnosis":2,"evidence":2,"mathematics":1,"method":"F2.2","pedagogy":1,"reviewer":"R3"},{"case_id":"dev_011","critical_error":1,"diagnosis":1,"evidence":2,"mathematics":0,"method":"F2.1","pedagogy":1,"reviewer":"R3"},{"case_id":"dev_014","critical_error":1,"diagnosis":0,"evidence":1,"mathematics":0,"method":"F2.2","pedagogy":1,"reviewer":"R3"},{"case_id":"dev_016","critical_error":0,"diagnosis":1,"evidence":2,"mathematics":1,"method":"F2.2","pedagogy":1,"reviewer":"R3"},{"case_id":"dev_027","critical_error":1,"diagnosis":1,"evidence":1,"mathematics":0,"method":"F2.2","pedagogy":1,"reviewer":"R3"},{"case_id":"dev_004","critical_error":0,"diagnosis":2,"evidence":2,"mathematics":2,"method":"F2.1","pedagogy":2,"reviewer":"R3"},{"case_id":"dev_019","critical_error":0,"diagnosis":2,"evidence":2,"mathematics":2,"method":"F1","pedagogy":2,"reviewer":"R3"},{"case_id":"dev_020","critical_error":0,"diagnosis":2,"evidence":2,"mathematics":2,"method":"F2.1","pedagogy":2,"reviewer":"R3"},{"case_id":"dev_014","critical_error":0,"diagnosis":2,"evidence":2,"mathematics":2,"method":"F1","pedagogy":1,"reviewer":"R3"},{"case_id":"dev_027","critical_error":1,"diagnosis":1,"evidence":1,"mathematics":0,"method":"F2.1","pedagogy":1,"reviewer":"R3"},{"case_id":"dev_024","critical_error":1,"diagnosis":0,"evidence":1,"mathematics":0,"method":"F1","pedagogy":0,"reviewer":"R3"},{"case_id":"dev_020","critical_error":0,"diagnosis":2,"evidence":2,"mathematics":2,"method":"F2.2","pedagogy":2,"reviewer":"R3"},{"case_id":"dev_011","critical_error":0,"diagnosis":2,"evidence":2,"mathematics":2,"method":"F2.2","pedagogy":2,"reviewer":"R3"},{"case_id":"dev_014","critical_error":0,"diagnosis":2,"evidence":2,"mathematics":1,"method":"F2.1","pedagogy":1,"reviewer":"R3"},{"case_id":"dev_027","critical_error":1,"diagnosis":1,"evidence":1,"mathematics":0,"method":"F1","pedagogy":0,"reviewer":"R3"},{"case_id":"dev_004","critical_error":0,"diagnosis":2,"evidence":2,"mathematics":2,"method":"F2.2","pedagogy":2,"reviewer":"R3"},{"case_id":"dev_007","critical_error":1,"diagnosis":1,"evidence":1,"mathematics":0,"method":"F2.2","pedagogy":1,"reviewer":"R3"},{"case_id":"dev_003","critical_error":0,"diagnosis":2,"evidence":2,"mathematics":1,"method":"F2.1","pedagogy":2,"reviewer":"R3"},{"case_id":"dev_011","critical_error":1,"diagnosis":0,"evidence":1,"mathematics":0,"method":"F1","pedagogy":0,"reviewer":"R3"},{"case_id":"dev_016","critical_error":0,"diagnosis":0,"evidence":1,"mathematics":1,"method":"F2.1","pedagogy":1,"reviewer":"R3"},{"case_id":"dev_020","critical_error":0,"diagnosis":1,"evidence":2,"mathematics":2,"method":"F1","pedagogy":2,"reviewer":"R3"},{"case_id":"dev_012","critical_error":1,"diagnosis":2,"evidence":1,"mathematics":0,"method":"F2.1","pedagogy":2,"reviewer":"R3"}]}''')
print(f"Loaded {len(DATA['heldout'])} operational records, "
      f"{len(DATA['expert'])} expert-review allocations, and "
      f"{len(DATA['development'])} development ratings.")


<div style="border:1px solid #d8e0ea;border-left:4px solid #4c6b8a;border-radius:7px;background:#f7f9fc;padding:10px 14px;margin:13px 0 8px 0;">
<strong>03 | Validate the evaluation partitions</strong><div style="color:#46596d;margin-top:5px;font-size:0.94em;">Verify paired case identifiers, expert ratings, missing drafts, and the development review structure.</div>
</div>

In [ ]:
# Validate dataset identity, matched-pair structure and rating completeness.
heldout = DATA["heldout"]
expert = DATA["expert"]
development = DATA["development"]
methods = ("F1", "F2.1")
case_ids = sorted(set(row["case_id"] for row in heldout))
assert len(case_ids) == 76 and len(heldout) == 152
assert all(sum(r["case_id"] == c and r["method"] == m for r in heldout) == 1
           for c in case_ids for m in methods)
assert len(expert) == 60 and sum(r["available"] for r in expert) == 58
assert len(set(r["problem_code"] for r in expert)) == 30
assert len(development) == 105 and len(set(r["reviewer"] for r in development)) == 3

expert_by_problem = defaultdict(dict)
for row in expert:
    expert_by_problem[row["problem_code"]][row["method"]] = row
paired = {p: d for p, d in expert_by_problem.items()
          if all(d.get(m, {}).get("available", False) for m in methods)}
assert len(paired) == 28

score_fields = ("diagnosis", "mathematics", "evidence", "pedagogy")
for row in expert:
    for field in score_fields + ("critical_error",):
        assert (row[field] is not None) == row["available"]
        if row["available"]:
            assert row[field] in ((0, 1) if field == "critical_error" else (0, 1, 2))

heldout_by_method = {m: {r["case_id"]: r for r in heldout if r["method"] == m}
                     for m in methods}
expert_all_available = {m: [r for r in expert if r["method"] == m and r["available"]]
                        for m in methods}
paired_rows = {m: [paired[p][m] for p in sorted(paired)] for m in methods}

def total_score(row):
    return sum(row[f] for f in score_fields)

print("Checked: 76 paired operational cases; 30 selected expert-review cases;")
print("28 matched expert-scored problem pairs and 2 additional F2.1-only scores.")
# Missing expert ratings are retained as unavailable, not assigned a score of zero.
assert sum(not r['available'] for r in expert) == 2
assert len({(r['case_id'], r['method']) for r in heldout}) == len(heldout)


## 1. Exploratory development comparison

The development-stage results are descriptive. They were not obtained from the held-out comparison and should not be pooled with it.


<div style="border:1px solid #d8e0ea;border-left:4px solid #4c6b8a;border-radius:7px;background:#f7f9fc;padding:10px 14px;margin:13px 0 8px 0;">
<strong>04 | Exploratory development ratings (Figure 9.2)</strong><div style="color:#46596d;margin-top:5px;font-size:0.94em;">Calculate the mean 0–8 expert score for F1, F2.1, and F2.2 on the 11 common development problems.</div>
</div>

In [ ]:
methods_dev=("F1","F2.1","F2.2")
dev_by=defaultdict(dict)
for r in development:
    dev_by[(r["case_id"],r["reviewer"])][r["method"]]=r
common_dev=sorted({c for c,_ in dev_by
                   if all(all(m in dev_by[(c,r)] for m in methods_dev)
                          for r in ("R1","R2","R3"))})
assert len(common_dev)==11

dev_case_mean={}
for m in methods_dev:
    dev_case_mean[m]=np.array([
        np.mean([total_score(dev_by[(case,r)][m]) for r in ("R1","R2","R3")])
        for case in common_dev],dtype=float)

dev_means={m:float(np.mean(v)) for m,v in dev_case_mean.items()}
fig,ax=plt.subplots(figsize=(7.5,4.1))
for i,m in enumerate(methods_dev):
    y=dev_means[m]
    ax.bar(i,y,width=.53,color=COLORS[m],alpha=.91)
    ax.text(i,y+.09,f"{y:.2f}",ha="center",fontweight="bold")
ax.set(xticks=np.arange(3),xticklabels=methods_dev,ylim=(0,8),
       ylabel="Mean total expert score (0–8)",
       title="Exploratory three-system development comparison")
style_grid(ax)
fig.text(.14,.01,"11 shared development problems; 3 development review forms; not held-out evidence.",fontsize=8.5)
fig.tight_layout(rect=[0,.055,1,1])
export_figure(fig,"Fig07_exploratory_development_ratings")


## 2. Held-out operational results

The 76 paired cases evaluate response availability, processing time, and arithmetic-tool activity. These operational outcomes do not establish mathematical or pedagogical correctness.


<div style="border:1px solid #d8e0ea;border-left:4px solid #4c6b8a;border-radius:7px;background:#f7f9fc;padding:10px 14px;margin:13px 0 8px 0;">
<strong>05 | Paired output coverage (Figure 9.3)</strong><div style="color:#46596d;margin-top:5px;font-size:0.94em;">Compare complete diagnosis-and-feedback drafts across all 76 problems using the exact McNemar test.</div>
</div>

In [ ]:
coverage = {m: sum(r["diagnosis_available"] and r["feedback_available"]
                   for r in heldout_by_method[m].values()) for m in methods}
assert coverage == {"F1": 70, "F2.1": 76}
def has_complete_draft(record):
    return bool(record['diagnosis_available'] and record['feedback_available'])

discordant_21 = sum(
    not has_complete_draft(heldout_by_method['F1'][c])
    and has_complete_draft(heldout_by_method['F2.1'][c])
    for c in case_ids
)
discordant_1 = sum(
    has_complete_draft(heldout_by_method['F1'][c])
    and not has_complete_draft(heldout_by_method['F2.1'][c])
    for c in case_ids
)
assert (discordant_21, discordant_1) == (6, 0)
p_coverage = binomtest(min(discordant_21, discordant_1),
                     discordant_21 + discordant_1, 0.5).pvalue

fig, ax = plt.subplots(figsize=(7.1, 3.4))
for idx, m in enumerate(methods):
    ax.barh(idx, coverage[m], height=0.48, color=COLORS[m])
    ax.text(coverage[m] + 0.8, idx,
            f"{coverage[m]}/76  ({100 * coverage[m]/76:.1f}%)",
            va="center", fontsize=10, fontweight="bold")
ax.set(yticks=[0,1], yticklabels=["F1", "F2.1"], xlim=(0,89),
       xlabel="Problems with diagnosis and feedback draft",
       title="Output coverage on the frozen held-out test")
ax.invert_yaxis()
ax.set_xticks([0,20,40,60,76])
style_grid(ax, "x")
fig.text(0.13, 0.00,
         f"Paired discordance: 6 F2.1-only / 0 F1-only; exact McNemar p = {p_coverage:.3f}.  "
         "All outputs required teacher review.", fontsize=8.2)
fig.tight_layout(rect=[0,0.06,1,1])
export_figure(fig, "Fig01_heldout_response_coverage")


## 3. Independent expert assessment

A single independent expert rated the available drafts on diagnosis, mathematics, evidence, and pedagogy (0–2 each). All direct system comparisons use the **28 problems with a usable response from both systems**. The two additional F2.1-only ratings are excluded from matched comparisons.


<div style="border:1px solid #d8e0ea;border-left:4px solid #4c6b8a;border-radius:7px;background:#f7f9fc;padding:10px 14px;margin:13px 0 8px 0;">
<strong>06 | Four rubric dimensions (Figure 9.4)</strong><div style="color:#46596d;margin-top:5px;font-size:0.94em;">Plot matched-pair means. The error bars here use paired-problem bootstrap 95% intervals calculated from the embedded scores.</div>
</div>

In [ ]:
labels = ("Diagnosis", "Mathematics", "Evidence", "Pedagogy")
values = {m: np.array([[r[f] for f in score_fields] for r in paired_rows[m]], dtype=float)
          for m in methods}
assert all(x.shape == (28,4) for x in values.values())
random = np.random.default_rng(RANDOM_SEED)
index = random.integers(0,28,size=(12000,28))
means = {m: arr.mean(axis=0) for m, arr in values.items()}
bounds = {m: np.quantile(arr[index,:].mean(axis=1),[0.025,0.975],axis=0)
          for m,arr in values.items()}
fig, ax = plt.subplots(figsize=(8.0,4.25))
x = np.arange(4); width = 0.35
for i,m in enumerate(methods):
    offset = (-0.5 if i==0 else 0.5) * width
    mu = means[m]; ci = bounds[m]
    bars = ax.bar(x + offset, mu, width=width, color=COLORS[m], label=m,
                  yerr=np.vstack([mu-ci[0],ci[1]-mu]), capsize=3,
                  error_kw={"elinewidth":1.1,"ecolor":"#39434B"})
    for bar,v in zip(bars,mu):
        ax.text(bar.get_x()+bar.get_width()/2, v+0.145,
                f"{v:.2f}", ha="center",fontsize=8.5)
ax.set(xticks=x,xticklabels=labels,ylim=(0,2.25),ylabel="Mean expert rating (0–2)",
       title="Expert-rated quality on 28 matched held-out problems")
ax.legend(frameon=False,loc="upper right")
style_grid(ax)
fig.tight_layout()
export_figure(fig,"Fig02_expert_rubric_comparison")


<div style="border:1px solid #d8e0ea;border-left:4px solid #4c6b8a;border-radius:7px;background:#f7f9fc;padding:10px 14px;margin:13px 0 8px 0;">
<strong>07 | Expert-identified critical errors (Figure 9.5)</strong><div style="color:#46596d;margin-top:5px;font-size:0.94em;">Compare the 28 paired critical-error judgements using an exact McNemar test.</div>
</div>

In [ ]:
crit={m:sum(r["critical_error"] for r in paired_rows[m]) for m in methods}
assert crit=={"F1":14,"F2.1":11}
crit_discordant_f1=sum(paired[p]["F1"]["critical_error"] == 1
                       and paired[p]["F2.1"]["critical_error"] == 0 for p in paired)
crit_discordant_f21=sum(paired[p]["F1"]["critical_error"] == 0
                        and paired[p]["F2.1"]["critical_error"] == 1 for p in paired)
p_critical=binomtest(min(crit_discordant_f1,crit_discordant_f21),
                    crit_discordant_f1+crit_discordant_f21,0.5).pvalue
assert sum(r["critical_error"] for r in expert if r["available"])==26

fig, ax=plt.subplots(figsize=(7.0,3.9))
xx=np.arange(2)
noncrit=np.array([28-crit[m] for m in methods])
critical=np.array([crit[m] for m in methods])
ax.bar(xx, noncrit, width=.52, color=PALE, label="No critical error flagged")
ax.bar(xx, critical, bottom=noncrit, width=.52,color=RED,label="Critical error flagged")
for i,m in enumerate(methods):
    ax.text(i,noncrit[i]/2,str(noncrit[i]),ha="center",va="center",fontweight="bold")
    ax.text(i,noncrit[i]+critical[i]/2,str(critical[i]),ha="center",va="center",color="white",fontweight="bold")
    ax.text(i,29,f"{100*crit[m]/28:.1f}% flagged",ha="center",fontsize=9)
ax.set(xticks=xx,xticklabels=methods,ylim=(0,32.5),ylabel="Scored outputs (n = 28 per system)",
       title="Critical errors in matched expert-rated outputs")
ax.legend(frameon=False,loc="lower center",bbox_to_anchor=(0.5,-0.27),ncol=2,fontsize=8.6)
style_grid(ax)
fig.text(.13,.00, f"Discordant pairs: {crit_discordant_f1} F1-only / {crit_discordant_f21} F2.1-only; "
         f"exact McNemar p = {p_critical:.3f}.",fontsize=8.2)
fig.tight_layout(rect=[0,0.1,1,1])
export_figure(fig,"Fig03_paired_critical_error_comparison")


<div style="border:1px solid #d8e0ea;border-left:4px solid #4c6b8a;border-radius:7px;background:#f7f9fc;padding:10px 14px;margin:13px 0 8px 0;">
<strong>08 | Paired total quality scores (Figure 9.6)</strong><div style="color:#46596d;margin-top:5px;font-size:0.94em;">Show within-problem differences, a Wilcoxon signed-rank test, and a bootstrap interval for the mean difference.</div>
</div>

In [ ]:
paired_ids=sorted(paired)
differences=np.array([total_score(paired[p]["F2.1"])-total_score(paired[p]["F1"])
                      for p in paired_ids],dtype=float)
w_stat=wilcoxon(differences,zero_method="wilcox",alternative="two-sided")
assert abs(w_stat.pvalue - 0.820) < 0.002
mean_diff=differences.mean()
idx=np.random.default_rng(RANDOM_SEED).integers(0,28,size=(20000,28))
diff_ci=np.quantile(differences[idx].mean(axis=1),[.025,.975])
order=np.argsort(differences,kind="stable")
fig, ax=plt.subplots(figsize=(7.5,7.1))
for pos,k in enumerate(order):
    value=differences[k]
    c=COLORS["F2.1"] if value>0 else COLORS["F1"] if value<0 else "#9AAAB6"
    ax.hlines(pos, 0, value, color=c,lw=2.0,alpha=.75)
    ax.plot(value,pos,"o",color=c,ms=5.2)
ax.axvline(0,color="#45545F",lw=1)
ax.set(yticks=np.arange(28),yticklabels=[paired_ids[k] for k in order],
       xlabel="Paired difference in total expert score (F2.1 minus F1)",
       xlim=(-8.8,9.0),title="Matched-problem variation in expert quality ratings")
ax.invert_yaxis(); ax.set_xticks(np.arange(-8,9,2))
style_grid(ax,"x")
fig.text(.16,.005,
         f"Mean paired difference: {mean_diff:+.2f}/8  "
         f"(bootstrap 95% CI [{diff_ci[0]:+.2f}, {diff_ci[1]:+.2f}]);  "
         f"Wilcoxon p = {w_stat.pvalue:.3f}.",fontsize=8.25)
fig.tight_layout(rect=[0,.055,1,1])
export_figure(fig,"Fig04_expert_paired_score_differences")


## 4. Execution time and computational verification

The arithmetic evaluator can confirm the value of a proposed expression, but cannot establish that the operation answers the mathematical question.


<div style="border:1px solid #d8e0ea;border-left:4px solid #4c6b8a;border-radius:7px;background:#f7f9fc;padding:10px 14px;margin:13px 0 8px 0;">
<strong>09 | Execution-time trade-off (Figure 9.7)</strong><div style="color:#46596d;margin-top:5px;font-size:0.94em;">Summarise wall-clock processing times and paired differences across the complete 76-problem test.</div>
</div>

In [ ]:
t1=np.array([heldout_by_method["F1"][c]["wall_seconds"] for c in case_ids])
t21=np.array([heldout_by_method["F2.1"][c]["wall_seconds"] for c in case_ids])
tdelta=t21-t1
assert round(t1.mean(),3)==91.420 and round(t21.mean(),3)==104.338
fig,(ax,ax2)=plt.subplots(1,2,figsize=(10.2,4.1),gridspec_kw={"width_ratios":[.9,1.25]})
box=ax.boxplot([t1,t21],patch_artist=True,
               widths=.45,medianprops={"color":"#1D2933","linewidth":1.4})
ax.set_xticks([1, 2], labels=['F1', 'F2.1'])
for patch,m in zip(box['boxes'],methods):patch.set_facecolor(COLORS[m]);patch.set_alpha(.45)
for i,v in enumerate([t1,t21],start=1):
    jitter=np.random.default_rng(RANDOM_SEED+i).uniform(-.13,.13,len(v))
    ax.scatter(i+jitter,v,s=12,color=COLORS[methods[i-1]],alpha=.33,edgecolors='none')
    ax.scatter([i],[v.mean()],marker='D',s=34,color=COLORS[methods[i-1]],zorder=5)
ax.set(ylabel="Wall-clock seconds per problem",title="Case-level runtime (n = 76)")
style_grid(ax)
ax2.hist(tdelta,bins=np.linspace(np.floor(tdelta.min()/10)*10,
                                 np.ceil(tdelta.max()/10)*10,13),
         color="#97C5C0",edgecolor="white")
ax2.axvline(0,color="#495866",lw=1.1,ls="--",label="Equal runtime")
ax2.axvline(tdelta.mean(),color=COLORS["F2.1"],lw=2,label=f"Mean: {tdelta.mean():+.1f} s")
ax2.set(xlabel="Paired runtime difference (F2.1 − F1), seconds",
        ylabel="Number of problems",title="Paired runtime cost")
style_grid(ax2)
ax2.legend(frameon=False,fontsize=8.4)
fig.tight_layout()
export_figure(fig,"Fig05_execution_time_tradeoff")


<div style="border:1px solid #d8e0ea;border-left:4px solid #4c6b8a;border-radius:7px;background:#f7f9fc;padding:10px 14px;margin:13px 0 8px 0;">
<strong>10 | Arithmetic-operation states (Figure 9.8)</strong><div style="color:#46596d;margin-top:5px;font-size:0.94em;">Count evaluated, rejected, not-attempted, and not-proposed operations without treating execution as correctness.</div>
</div>

In [ ]:
statuses = [("arithmetic_evaluated","Evaluated"),("rejected","Rejected"),
            ("not_attempted","Not attempted"),("not_proposed","Not proposed")]
status_colors={"Evaluated":"#087F78","Rejected":"#BE7360",
               "Not attempted":"#9BA7B5","Not proposed":"#D9E2E8"}
fig,ax=plt.subplots(figsize=(9.0,3.35))
for i,m in enumerate(methods):
    counts=Counter(r["arithmetic_status"] for r in heldout_by_method[m].values())
    assert sum(counts.values())==76
    left=0
    for raw,name in statuses:
        n=counts.get(raw,0)
        if n:
            ax.barh(i,n,left=left,height=.47,color=status_colors[name],
                    label=name if i==0 else "",edgecolor="white",linewidth=.7)
            if n>=5:ax.text(left+n/2,i,str(n),ha="center",va="center",fontsize=9,
                           color="white" if name in ("Evaluated","Rejected") else "#273842",fontweight="bold")
            else:ax.annotate(str(n),(left+n/2,i),xytext=(0,14),
                              textcoords="offset points",ha="center",fontsize=8.4)
        left+=n
ax.set(yticks=[0,1],yticklabels=list(methods),xlim=(0,80),
       xlabel="Problems (n = 76 per system)",
       title="Arithmetic operation states in the held-out experiment")
ax.invert_yaxis();style_grid(ax,"x")
# Manual legend includes all four categories, including categories absent from F2.1.
from matplotlib.patches import Patch
ax.legend(handles=[Patch(facecolor=status_colors[name],label=name) for _,name in statuses],
          frameon=False,loc="upper center",bbox_to_anchor=(.5,-.25),ncol=4,fontsize=8.3)
fig.tight_layout(rect=[0,0.06,1,1])
export_figure(fig,"Fig06_arithmetic_operation_states")


## 5. Statistical summary and supplementary checks

The tables below use denominators appropriate to each analysis. Coverage and quality should not be merged into one accuracy measure.


<div style="border:1px solid #d8e0ea;border-left:4px solid #4c6b8a;border-radius:7px;background:#f7f9fc;padding:10px 14px;margin:13px 0 8px 0;">
<strong>11 | Reproduce the numerical results</strong><div style="color:#46596d;margin-top:5px;font-size:0.94em;">Calculate coverage, expert-rating means, critical-error counts, test statistics, and development summaries.</div>
</div>

In [ ]:
print("HELD-OUT PERFORMANCE (76 CASES PER METHOD)")
print("-" * 67)
for m in methods:
    records=list(heldout_by_method[m].values())
    print(f"{m:5s}: coverage={coverage[m]}/76 ({coverage[m]/76:.1%}); "
          f"calls={sum(r['model_calls'] for r in records)}; "
          f"mean runtime={np.mean([r['wall_seconds'] for r in records]):.3f} s; "
          f"review route={sum(r['review_required'] for r in records)}/76; "
          f"auto-release={sum(r['automatic_release'] for r in records)}")
print(f"Exact McNemar (coverage): p={p_coverage:.6f}")

print("\nEXPERT QUALITY (28 MATCHED HELD-OUT PAIRS)")
print("-" * 67)
for m in methods:
    rows=paired_rows[m]
    mean_fields=" / ".join(f"{np.mean([r[f] for r in rows]):.3f}" for f in score_fields)
    print(f"{m:5s}: Diagnosis/Math/Evidence/Pedagogy = {mean_fields}; "
          f"total={np.mean([total_score(r) for r in rows]):.3f}/8; "
          f"critical={crit[m]}/28")
print(f"Paired mean total difference (F2.1-F1): {mean_diff:+.3f}/8")
print(f"Paired Wilcoxon (two-sided): p={w_stat.pvalue:.6f}")
print(f"Exact McNemar (critical error): p={p_critical:.6f}")
print("Full scored sample: 26 critical errors / 58 available outputs.")
print("\nEXPLORATORY DEVELOPMENT (11 MATCHED CASES, 3 REVIEW FORMS)")
for m in methods_dev:
    print(f"{m:5s}: mean total score={dev_means[m]:.3f}/8")


### Post hoc sensitivity analysis

The source notebook flags six review problems as potentially ambiguous or inconsistent. Excluding them is an **exploratory sensitivity check**, not a replacement for the full prespecified 28-pair analysis. The original problem texts are not available here to independently assess those flags.


<div style="border:1px solid #d8e0ea;border-left:4px solid #4c6b8a;border-radius:7px;background:#f7f9fc;padding:10px 14px;margin:13px 0 8px 0;">
<strong>12 | Sensitivity to flagged problem statements</strong><div style="color:#46596d;margin-top:5px;font-size:0.94em;">Compare the mean paired score difference before and after excluding six flagged review problems.</div>
</div>

In [ ]:
flagged={"P02","P05","P10","P19","P23","P30"}
clean_pairs=[p for p in paired_ids if p not in flagged]
posthoc=np.array([total_score(paired[p]["F2.1"])-total_score(paired[p]["F1"])
                 for p in clean_pairs],dtype=float)
print(f"Primary matched sample: {len(paired_ids)} problems, mean difference {mean_diff:+.3f}/8")
print(f"Post hoc subset excluding flagged prompts: {len(clean_pairs)} problems, "
      f"mean difference {posthoc.mean():+.3f}/8")
print("This subset is descriptive and was not precommitted.")


## 6. Interpretation and figure provenance

| Result | Interpretation |
|---|---|
| Higher output coverage | More problems yield reviewable drafts; this does not prove higher accuracy. |
| Higher rubric score | One expert judged the available draft more favourably on a specified dimension. |
| Critical-error flag | Expert identified a consequential mathematical or pedagogical concern. |
| Arithmetic evaluated | A proposed arithmetic expression was computed; its semantic suitability remains unverified. |
| Teacher-review routing | Generated material is not automatically released to learners. |

**Reporting limits:** The held-out expert review uses one rater and 28 matched problems. Non-significant paired tests do not establish equivalence. Figures are reconstructed from the records supplied with the source notebook; the original sealed inputs and scoring workbooks are not available for an independent provenance check. The newly computed confidence intervals shown in the rubric plot are specifically defined here; they should not be described as verified methodology for any earlier figure with undocumented error bars.


<div style="border:1px solid #d8e0ea;border-left:4px solid #4c6b8a;border-radius:7px;background:#f7f9fc;padding:10px 14px;margin:13px 0 8px 0;">
<strong>13 | Audit figure outputs and analysis records</strong><div style="color:#46596d;margin-top:5px;font-size:0.94em;">Check the seven exported figures in each format and calculate a checksum for the included numerical records.</div>
</div>

In [ ]:
# Identify the embedded analytical tabulation without exposing private file names.
canonical = json.dumps(DATA, sort_keys=True, separators=(',', ':'))
analysis_hash = hashlib.sha256(canonical.encode('utf-8')).hexdigest()
print('Embedded analysis records SHA-256:', analysis_hash)
print('Output directory:', OUTPUT_DIR.resolve())
for suffix in ('png', 'svg', 'pdf'):
    figures = sorted(OUTPUT_DIR.glob(f'Fig*.{suffix}'))
    assert len(figures) == 7, f'Expected seven {suffix} files, found {len(figures)}.'
    print(f'{suffix.upper()}:', len(figures), 'figures')
print('All seven figures generated. The checksum identifies the embedded tabulation only.')
